# 02a — Vectorized phenotype extraction

## Purpose
Extract configured clinical phenotypes from problems, diagnoses, and prescribing data, evaluate their temporal windows, aggregate composite phenotypes, and save reusable parquet artifacts.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd
from pathlib import Path

from icare_risk.utils.paths import (
    get_path, get_data_root_from_schema,
    print_paths_summary, print_directory_content
)

# Define schema configuration to use.
SCHEMA_CFG = 'configs/schema_1000p.yaml'
PHENOTYPES_CFG = 'configs/phenotypes.yaml'

# Paths
DATA_DIR = get_data_root_from_schema(schema_path=get_path(SCHEMA_CFG)) 
DATA_PROC_DIR = DATA_DIR / 'processed'
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Debug
print(f"{'Current Working Directory:':<26} {os.getcwd()}")
print(f"{'Data Path:':<26} {DATA_DIR}")
print(f"{'Data Processed Path:':<26} {DATA_PROC_DIR}")
paths_map = print_paths_summary(raise_on_missing=False)
print_directory_content(dir=DATA_DIR)


Current Working Directory: /app
Data Path:                 /app/data/synthetic/10000p
Data Processed Path:       /app/data/synthetic/10000p/processed

================= ICARE_RISK WORKSPACE PATHS SUMMARY =================
  [✓] Found    📁 DIR  | Project Root          : /app
  [✓] Found    📁 DIR  | Config Directory      : /app/configs
  [✓] Found    📁 DIR  | Data Directory        : /app/data
  [✓] Found    📁 DIR  | Output Directory      : /app/outputs
  [✓] Found    📄 FILE | Database Config       : /app/configs/generate_db.yaml
  [✓] Found    📄 FILE | Code Search Config    : /app/configs/code_search.yaml
  [✗] MISSING  📄 FILE | Schema YAML           : /app/configs/schema.yaml
  [✓] Found    📄 FILE | Analysis YAML         : /app/configs/analysis.yaml
  [✓] Found    📄 FILE | Phenotypes YAML       : /app/configs/phenotypes.yaml
  [✓] Found    📄 FILE | Lookup 3GCR           : /app/configs/lookups/3GCR.csv
  [✓] Found    📄 FILE | Lookup 3GCS           : /app/configs/lookups/3GCS.csv
⚠️ Warni

PosixPath('/app/data/synthetic/10000p')

In [3]:
# Convert to parquets if needed
from icare_risk.utils.conversion import convert
convert(DATA_DIR, 'csv_to_parquet')

Converting: icare_episodes_anon.csv -> icare_episodes_anon.parquet
Converting: icare_episodes_diagnosis_anon.csv -> icare_episodes_diagnosis_anon.parquet
Converting: icare_demographic_anon.csv -> icare_demographic_anon.parquet
Converting: icare_microbiology_anon.csv -> icare_microbiology_anon.parquet
Converting: icare_problems_anon.csv -> icare_problems_anon.parquet
Converting: icare_pharmacy_prescribing_anon.csv -> icare_pharmacy_prescribing_anon.parquet
Converting: icare_pharmacy_administration_anon.csv -> icare_pharmacy_administration_anon.parquet
Converting: icare_vital_signs_anon.csv -> icare_vital_signs_anon.parquet
Converting: icare_pathology_blood_anon.csv -> icare_pathology_blood_anon.parquet
Converting: table1_demographics.csv -> table1_demographics.parquet
Successfully converted 10 file(s).


## Load source tables

In [4]:
def load_from_pandas_manual(DATA_DIR):
    """Load from pandas"""
    # Load data
    df_episodes = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")                # parse_dates=["ADMISSION_DATE"],
    df_problems = pd.read_parquet(DATA_DIR / "icare_problems_anon.parquet")                # parse_dates=["PROBLEM_DT_TM"],
    df_diagnoses = pd.read_parquet(DATA_DIR / "icare_episodes_diagnosis_anon.parquet")     # parse_dates=["DIAGNOSIS_DATE"],
    df_prescribing = pd.read_parquet(DATA_DIR / "icare_pharmacy_prescribing_anon.parquet") # parse_dates=["ORDER_DT_TM"],

    # Convert string columns back to proper date-times
    df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
    df_problems["PROBLEM_DT_TM"] = pd.to_datetime(df_problems["PROBLEM_DT_TM"])
    df_diagnoses["DIAGNOSIS_DATE"] = pd.to_datetime(df_diagnoses["DIAGNOSIS_DATE"])
    df_prescribing["ORDER_DT_TM"] = pd.to_datetime(df_prescribing["ORDER_DT_TM"])

    return {
        'episodes': df_episodes, 
        'problems': df_problems, 
        'diagnosies': df_diagnoses, 
        'prescirbing': df_prescribing
    }

#dfs =  load_from_pandas(DATA_DIR)

In [ ]:
# --- OPTION A: Load using pandas ---
from icare_risk.clinphen.io.loaders import load_from_pandas

domains = ['episodes', 'problems', 'diagnosis', 'prescribing']

# Load dataframes
dfs = load_from_pandas(
    schema_path=SCHEMA_CFG,
    domains_to_load=domains,
    verbose=1
)

✅ Schema validation passed! All files exist and all physical columns match successfully.
Loaded... episodes (2.46MB) shape (10000, 11)
Loaded... problems (6.57MB) shape (29970, 6)
Loaded... diagnosis (11.61MB) shape (29907, 12)
Loaded... prescribing (47.26MB) shape (70112, 16)


In [ ]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
domains = ['episodes', 'problems', 'diagnosis', 'prescribing']

# Load dataframes
dfs = load_from_duckdb(
    schema_path=SCHEMA_CFG,
    phenotype_path=PHENOTYPES_CFG,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = dfs['episodes']
df_problems = dfs['problems']
df_diagnoses = dfs['diagnosis']
df_prescribing = dfs['prescribing']

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Loaded... episodes (2.46MB) shape (10000, 11)
Loaded... problems (0.50MB) shape (2230, 6) | Filtered codes: 979
Loaded... diagnosis (0.00MB) shape (0, 12) | Filtered codes: 13
Loaded... prescribing (13.56MB) shape (20103, 16) | Filtered codes: 39


In [7]:
from icare_risk.utils.diagnostics import check_df

# Brief check
check_df(
    df_episodes,
    "Episodes input",
    columns=[
        "SUBJECT",
        "SPELL_IDENTIFIER",
        "ADMISSION_DATE",
        "DISCHARGE_DATE",
    ],
    n=5,
)


CHECK: Episodes input
Shape: 10,000 rows x 11 columns

Preview:


,SUBJECT,SPELL_IDENTIFIER,ADMISSION_DATE,DISCHARGE_DATE
0,10001,7896381,2020-09-29,2020-10-07
1,10002,1805970,2020-08-03,2020-08-15
2,10003,6698303,2022-11-02,2022-11-11
3,10004,6804249,2023-01-05,2023-01-10
4,10005,2301493,2020-11-13,2020-12-04


## Configure domains and phenotypes

In [8]:
from icare_risk.utils.io import load_yaml_file
from icare_risk.utils.paths import get_path

domain_registry = {
    "prescribing": {
        "df": df_prescribing,
        "subject_col": "SUBJECT",
        "time_col": "ORDER_DT_TM",
        "target_col": "MEDICATION_NAME",
        "match_type": "contains",
    },
    "problems": {
        "df": df_problems,
        "subject_col": "SUBJECT",
        "time_col": "PROBLEM_DT_TM",
        "target_col": "PROBLEM_CODE",
    },
    "diagnosis": {
        "df": df_diagnoses,
        "subject_col": "SUBJECT",
        "time_col": "DIAGNOSIS_DATE",
        "sources": [
            {"target_col": "DIAGNOSIS_CODE_SNOMED", "match_type": "exact"},
            {"target_col": "DIAGNOSIS_CODE_ICD",    "match_type": "prefix"},
        ],
    },
}

# Load from pkg files.
#all_phenotypes = load_pkg_yaml("config/icare/phenotypes.yaml")

# Load phenotypes and filter
all_phenotypes = load_yaml_file(get_path(PHENOTYPES_CFG))
charlson = {k:v for k,v in all_phenotypes.items() if k.startswith('charlson_')}
phenotypes = all_phenotypes  # Replace with a filtered dictionary for a smaller run.

print(f"Configured phenotypes: {len(phenotypes):,}")

Configured phenotypes: 66


## Extract and window events

In [10]:
# Libraries
from icare_risk.clinphen.utils.vectorized import (
    run_vectorized,
    evaluate_temporal_windows,
    aggregate_composite_phenotypes,
    parse_phenotypes_to_batches
)


# 1. Extract raw historical events
df_events = run_vectorized(
    domains=domain_registry,
    phenotypes=phenotypes,
    verbose=1,
)

# 2. Create index_admission column
df_episodes["index_admission"] = pd.to_datetime(
    df_episodes["ADMISSION_DATE"].astype(str) + " "
    + df_episodes["ADMISSION_TIME"].astype(str),
    errors="coerce",
)

# 3. Build the window configuration mapping from phenotypes YAML dictionary
window_configs = {}
for batch in parse_phenotypes_to_batches(phenotypes):
    for sub_event_name, config in batch["configs"].items():
        if config.get("window") is not None:
            window_configs[sub_event_name] = config["window"]

# 4. Filter events given the episode and the windows
df_events_windowed = evaluate_temporal_windows(
    episodes_df=df_episodes,
    events_df=df_events,
    window_configs=window_configs,
    subject_col="SUBJECT",
    admission_col="index_admission",
    event_col="event_name",
    event_time_col="event_dt_tm",
    prefix="hx_",
)

# 5. Aggregate component-level binary indications into parent clinical phenotypes
df_aggregated = aggregate_composite_phenotypes(
    df=df_events_windowed,
    phenotype_configs=phenotypes,
    prefix="",
    drop_components=True,
)

print(f"Raw events:       {df_events.shape}")
print(f"Windowed events:  {df_events_windowed.shape}")
print(f"Aggregated output:{df_aggregated.shape}")


--- Running Vectorized Bench ---

--- 1. Parsing Phenotypes YAML ---
Generated 7 execution batch(es).

--- 2. Executing Batches ---

--- DEBUG BATCH 1 (Source 1) [problems -> PROBLEM_CODE] ---
Match Type: exact, Case Sensitive: False
YAML is looking for (first 5):
  - charlson_hx_mi__comp0
  - charlson_hx_mi__comp1
  - charlson_hx_chf__comp0
  - charlson_hx_chf__comp1
  - charlson_hx_pvd__comp0
DataFrame top 5 most common values (with counts):
  - 44054006: 424 occurrence(s)
  - 73211009: 166 occurrence(s)
  - 13645005: 113 occurrence(s)
  - 11687002: 109 occurrence(s)
  - 52448006: 91 occurrence(s)
-> Matched 2363 record(s) using column 'PROBLEM_CODE' with match_type='exact'.

--- DEBUG BATCH 2 (Source 1) [prescribing -> MEDICATION_NAME] ---
Match Type: contains, Case Sensitive: False
YAML is looking for (first 5):
  - charlson_hx_chf__comp2
  - charlson_hx_diabetes_uncomp__comp2
  - gavaghan_hx_prior_antibiotics_90d__comp0
  - pitt_hypo_score_rule__comp1
DataFrame top 5 most common 

In [11]:
# Local Libraries
from icare_risk.clinphen.scores.gavaghan import compute_gavaghan_score
from icare_risk.clinphen.scores.holmgren import compute_holmgren_score
from icare_risk.clinphen.scores.charlson import compute_charlson_score
from icare_risk.clinphen.scores.jones import compute_jones_score
from icare_risk.clinphen.scores.tumbarello import compute_tumbarello_score
from icare_risk.clinphen.scores.kim import compute_kim_score

# Compute scores
df_aggregated['charlson_score'] = compute_charlson_score(df_aggregated, verbose=0)
df_aggregated['gavaghan_score'] = compute_gavaghan_score(df_aggregated, verbose=0)
df_aggregated['holmgren_score'] = compute_holmgren_score(df_aggregated, verbose=0)
df_aggregated['jones_score'] = compute_jones_score(df_aggregated, verbose=0)
df_aggregated['tumbarello_score'] = compute_tumbarello_score(df_aggregated, verbose=0)
df_aggregated['kim_score'] = compute_kim_score(df_aggregated, verbose=0)

# Show
score_cols = [
    'charlson_score', 'gavaghan_score', 'holmgren_score', 
    'jones_score', 'tumbarello_score', 'kim_score'
]

df_scores_dist = (
    df_aggregated[score_cols]
    .apply(pd.Series.value_counts)
    .sort_index()     
    .fillna(0)        
    .astype(int)     
)
display(df_scores_dist)

,charlson_score,gavaghan_score,holmgren_score,jones_score,tumbarello_score,kim_score
0.0,2405,9993,10000,9791,9377,9377
1.0,2327,7,0,0,4,590
2.0,1378,0,0,209,586,3
3.0,1241,0,0,0,3,0
4.0,1723,0,0,0,28,0
5.0,816,0,0,0,0,28
6.0,64,0,0,0,2,2
7.0,26,0,0,0,0,0
8.0,6,0,0,0,0,0
9.0,5,0,0,0,0,0


## Save artifacts

In [12]:
from icare_risk.clinphen.utils.io import save_phenotype_artifacts

save_phenotype_artifacts(
    df_raw=df_events,
    df_windowed=df_events_windowed,
    df_aggregated=df_aggregated,
    output_dir=DATA_PROC_DIR,
    prefix="02a_derived_",
)


--- Saving Phenotype Artifacts ---
 Saved Raw extracted matching events (82,598 rows)
   └─ Path: /app/data/synthetic/10000p/processed/02a_derived_events_extracted_raw.parquet
 Saved Window-filtered events (10,000 rows)
   └─ Path: /app/data/synthetic/10000p/processed/02a_derived_events_extracted_windowed.parquet
 Saved Aggregated patient/spell phenotypes (10,000 rows)
   └─ Path: /app/data/synthetic/10000p/processed/02a_derived_phenotypes_aggregated.parquet
-----------------------------------



## Visually debugging with total counts

In [13]:
def event_summary_v1(df_events):
    return (
        df_events.groupby("event_name")
        .agg(
            total_records=("SUBJECT", "count"),
            unique_patients=("SUBJECT", "nunique"),
        )
        .reset_index()
        .sort_values("unique_patients", ascending=False)
    ).to_string(index=False)

def event_summary_v2(df_events_windowed):
    # 1. Identify all columns starting with 'hx_'
    hx_cols = [col for col in df_events_windowed.columns if col.startswith("hx_")]

    # 2. Group by SUBJECT, check if any row per patient is True/1, then sum
    return (
        df_events_windowed[["SUBJECT"] + hx_cols]
        .groupby("SUBJECT")
        .any()  # True if the patient has a 1 in at least one episode
        .sum()  # Counts total unique patients with True per column
        .reset_index(name="patient_count")
        .rename(columns={"index": "phenotype"})
        .sort_values("patient_count", ascending=False)
    ).to_string(index=False)

In [14]:
print(event_summary_v1(df_events))

                              event_name  total_records  unique_patients
     kim_hx_prior_antibiotics_90d__comp0          10022             6244
   jones_hx_prior_antibiotics_30d__comp1          10022             6244
    kim_hx_recent_antibiotics_90d__comp0          10022             6244
    kim_hx_recent_antibiotics_90d__comp1          10022             6244
     tumbarello_hx_recent_abx_90d__comp0          10022             6244
   jones_hx_prior_antibiotics_30d__comp0          10022             6244
gavaghan_hx_prior_antibiotics_90d__comp1          10022             6244
      charlson_hx_diabetes_uncomp__comp2          10081             6231
      charlson_hx_diabetes_uncomp__comp0            807              782
        charlson_hx_renal_mod_sev__comp0            246              239
               charlson_hx_stroke__comp0            193              190
            charlson_hx_pulmonary__comp0            166              165
             charlson_hx_dementia__comp0           

In [15]:
print(event_summary_v2(df_events_windowed))

                                  phenotype  patient_count
      hx_charlson_hx_diabetes_uncomp__comp2           3639
    hx_kim_hx_recent_antibiotics_90d__comp0            598
    hx_kim_hx_recent_antibiotics_90d__comp1            598
hx_gavaghan_hx_prior_antibiotics_90d__comp1            591
     hx_kim_hx_prior_antibiotics_90d__comp0            591
     hx_tumbarello_hx_recent_abx_90d__comp0            591
      hx_charlson_hx_diabetes_uncomp__comp0            403
   hx_jones_hx_prior_antibiotics_30d__comp0            209
   hx_jones_hx_prior_antibiotics_30d__comp1            209
        hx_charlson_hx_renal_mod_sev__comp0            130
               hx_charlson_hx_stroke__comp0             89
            hx_charlson_hx_pulmonary__comp0             80
             hx_charlson_hx_dementia__comp0             74
                  hx_charlson_hx_pud__comp0             50
                  hx_charlson_hx_chf__comp0             50
                hx_charlson_hx_rheum__comp0             

In [16]:
from icare_risk.utils.diagnostics import (
    check_df, check_value_counts, check_merge_keys,
)

check_value_counts(
    df_events, "event_name", name="Events by phenotype component", n=20,
)

check_value_counts(
    df_events, "domain", name="Events by domain", n=20,
)


Value check: Events by phenotype component


,count
event_name,
charlson_hx_diabetes_uncomp__comp2,10081
kim_hx_recent_antibiotics_90d__comp0,10022
jones_hx_prior_antibiotics_30d__comp1,10022
kim_hx_recent_antibiotics_90d__comp1,10022
kim_hx_prior_antibiotics_90d__comp0,10022
jones_hx_prior_antibiotics_30d__comp0,10022
gavaghan_hx_prior_antibiotics_90d__comp1,10022
tumbarello_hx_recent_abx_90d__comp0,10022
charlson_hx_diabetes_uncomp__comp0,807



Value check: Events by domain


,count
domain,
prescribing,80235
problems,2363
